<a id="top"></a>

# Demo: trace the round-trip, message by message

```yaml
title: "Demo: trace the round-trip, message by message"
keywords: round-trip, four messages, conversation history, call id, stateless, token cost, trace, teacher demo
estimated duration: 10
```

> **Lesson:** L07. Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). **No API key needed** — this demo dissects a *crafted transcript* of the exact exchange [Demo 2](L0704_lecture.ipynb) produces, so the message structure is identical every run. (You can instead replay the real `messages` list Demo 2 built.)
>
> The point to land: a single tool-using exchange is **at minimum four messages**, and the call id is the thread tying request to result. Every tool call grows the history — that is the protocol, not a side effect.

## Contents

- [1. Setup — a captured four-message transcript](#1-setup--a-captured-four-message-transcript)
- [2. Walk the four messages in order](#2-walk-the-four-messages-in-order)
- [3. The id ties the result to the request](#3-the-id-ties-the-result-to-the-request)
- [4. Count the growth](#4-count-the-growth)
- [5. Takeaways](#5-takeaways)

## 1. Setup — a captured four-message transcript

The same shape Demo 2 produced, written out as LangChain **message objects** so you can walk it slowly with no live call. Three message types appear: `HumanMessage`, `AIMessage` (carrying `tool_calls`), and `ToolMessage`.

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage

# A captured transcript of one successful round-trip (Demo 2's exchange), as the
# LangChain message objects Demo 2 accumulated in its `messages` list.
CALL_ID = "toolu_01ABC"  # the unique id the model assigned to its tool call

transcript: list[BaseMessage] = [
    HumanMessage("What is 18,374 multiplied by 92,431?"),
    AIMessage(
        content="",
        tool_calls=[
            {
                "id": CALL_ID,
                "name": "calculator",
                "args": {"expression": "18374 * 92431"},
                "type": "tool_call",
            }
        ],
    ),
    ToolMessage(content="1698367394", tool_call_id=CALL_ID),
    AIMessage(content="18,374 x 92,431 = 1,698,367,394."),
]
print(f"{len(transcript)} messages captured")

[↑ Back to top](#top)

## 2. Walk the four messages in order

Print each message with its **type** (`human`, `ai`, `tool`), what it carries, and **who produced it**. The user "asked once" — yet the history is four messages long.

In [ ]:
def summarize(msg: BaseMessage) -> str:
    """Describe what one message carries: text, a tool call, or a tool result."""
    if isinstance(msg, AIMessage) and msg.tool_calls:
        return "tool_calls -> " + ", ".join(c["name"] for c in msg.tool_calls)
    if isinstance(msg, ToolMessage):
        return f"tool result for id={msg.tool_call_id}"
    return f"text -> {msg.content!r}"


producers = ["the human", "the model", "the APPLICATION (a tool-role message)", "the model"]
for i, (msg, who) in enumerate(zip(transcript, producers, strict=True), start=1):
    print(f"Message {i} | type={msg.type:5} | {summarize(msg)}")
    print(f"           produced by {who}")

[↑ Back to top](#top)

## 3. The id ties the result to the request

The `ToolMessage` in Message 3 names the **same** id (`tool_call_id`) as the tool call in Message 2's `AIMessage`. That id is how the application says "this result answers *that* request" — essential once more than one call is in flight ([L10](L0702_lecture.md)).

In [ ]:
use_msg = transcript[1]  # the AIMessage carrying the tool call (message 2)
result_msg = transcript[2]  # the ToolMessage (message 3)
assert isinstance(use_msg, AIMessage) and isinstance(result_msg, ToolMessage)
print("tool call id   (msg 2):", use_msg.tool_calls[0]["id"])
print("tool result id (msg 3):", result_msg.tool_call_id)
assert use_msg.tool_calls[0]["id"] == result_msg.tool_call_id
print("ids match -> this result answers that exact request")

[↑ Back to top](#top)

## 4. Count the growth

The user asked **once**, but the conversation grew by **four** messages. Every future tool call repeats this growth — and the tool *definition* is re-sent on every request because the model is **stateless**.

In [ ]:
print("user questions asked     :", 1)
print("messages now in history  :", len(transcript))
print()
# Tools cost tokens TWICE OVER: the definition rides in every request, and the
# result lives in the history for every later turn. A 500-token definition across
# a 10-turn chat is ~5,000 input tokens before the tool is even called.
DEFINITION_TOKENS = 500
TURNS = 10
print(f"~{DEFINITION_TOKENS * TURNS} input tokens spent on the definition alone over {TURNS} turns")

[↑ Back to top](#top)

## 5. Takeaways

- A successful round-trip is **four messages**: `HumanMessage -> AIMessage(tool_calls) -> ToolMessage -> AIMessage(final)`. Four is the *minimum*, not a fixed number.
- The history carries **three message types** — `human`, `ai`, and a dedicated **`tool`** message. The `ToolMessage` is the application speaking, not the human; the type marks **protocol position**, not who is a person.
- The model is **stateless across calls** — Message 4 required re-sending Messages 1-3 *and* the tool definition. The model did not "remember" the tool.
- Tools cost tokens **twice over** — definition re-sent every request, result kept in history. Forward-link to [L13](L0702_lecture.md) (model power) and L16 (context management).

[↑ Back to top](#top)